# CS 415 — Ollama + Google Colab

Run a language model in Colab and call its protected API from your **own computer**.

```text
Colab → Ollama server → downloaded model
             ↑
       ngrok (HTTPS + Basic Authentication)
             ↑ request / ↓ response
       Local computer: test_ollama.py
```

Run the cells **in order, one at a time**. Stop and fix any error before continuing.
You need a Google account, an available Colab GPU, an ngrok account, and Python on
your computer. Your computer does not need a GPU or Ollama.

This is a short, interactive learning exercise. Colab resources are temporary and
not guaranteed; this notebook is not a permanent hosting service. Follow the
[Colab usage restrictions](https://research.google.com/colaboratory/faq.html#limitations-and-restrictions).
Keep working in the notebook while testing; do not use keep-alive tricks to bypass limits.

The exercise demonstrates HTTP requests and JSON responses, like the APIs studied
with FastAPI. Ollama already provides the API; no FastAPI wrapper or AI agent is needed.

## 1 — Choose the model and GPU runtime

Open **Runtime → Change runtime type**, select a **GPU**, and connect.
Leave `PROFILE = "student"` for the regular exercise.

| Profile | Model | Intended hardware | Approximate download |
|---|---|---|---|
| `student` | `qwen3.5:4b` | T4 or another suitable NVIDIA GPU | 4 GB |
| `small` | `qwen3.5:0.8b` | Smaller alternative for limited resources | 1–2 GB |
| `instructor` | `gpt-oss:120b` | One 80 GB-class GPU, such as A100 80 GB | 65 GB |

**An A100 label does not guarantee 80 GB.** A T4 or A100 40 GB does not meet this
notebook's 120B demonstration requirements. The next cell checks actual memory;
Colab may not offer suitable hardware, even on paid plans. In that case use a
student profile here and run the instructor demonstration on suitable hardware elsewhere.

Download size is not the same as inference memory: context and runtime overhead
also consume memory. The checks below are conservative lab guards, not universal
minimum requirements. If you change profiles later, run cleanup first and restart
from this step. Changing the runtime can discard all installed software and models.

In [ ]:
# @title Model profile
PROFILE = "student"  # @param ["student", "small", "instructor"]
PROFILES = {
    "student": {"model": "qwen3.5:4b", "num_ctx": 4096, "disk_gib": 10, "vram_gib": 6},
    "small": {"model": "qwen3.5:0.8b", "num_ctx": 4096, "disk_gib": 6, "vram_gib": 3},
    "instructor": {"model": "gpt-oss:120b", "num_ctx": 8192, "disk_gib": 80, "vram_gib": 75},
}
if PROFILE not in PROFILES:
    raise ValueError("Choose student, small, or instructor.")
CONFIG = PROFILES[PROFILE]
MODEL = CONFIG["model"]
NUM_CTX = CONFIG["num_ctx"]
OLLAMA_URL = "http://127.0.0.1:11434"
HARDWARE_CHECKED = False
print(f"Profile: {PROFILE} | Model: {MODEL} | Context: {NUM_CTX}")

## 2 — Check GPU memory and free disk space

This check stops if no working NVIDIA GPU is available or the selected profile
needs more memory/disk. Enable a GPU runtime or choose a smaller profile and rerun
from Step 1. These checks do not download or load a model.

In [ ]:
import shutil
import subprocess
from pathlib import Path

HARDWARE_CHECKED = False
if not Path("/content").is_dir():
    raise RuntimeError("Run this notebook in Google Colab; run only test_ollama.py locally.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU found. Select a GPU under Runtime → Change runtime type.")
try:
    gpu_info = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True, timeout=15,
    ).stdout.strip()
except (subprocess.SubprocessError, OSError):
    raise RuntimeError("The GPU is unavailable. Reconnect to a GPU runtime and rerun.") from None

free_vram = []
for line in gpu_info.splitlines():
    name, total_mib, free_mib = [part.strip() for part in line.split(",")]
    print(f"{name}: {float(total_mib)/1024:.1f} GiB total, {float(free_mib)/1024:.1f} GiB free")
    free_vram.append(float(free_mib) / 1024)
free_disk = shutil.disk_usage("/content").free / (1024 ** 3)
print(f"Free disk: {free_disk:.1f} GiB")
if not free_vram or max(free_vram) < CONFIG["vram_gib"]:
    raise RuntimeError(f"This profile needs at least {CONFIG['vram_gib']} GiB free on one GPU. Use a smaller profile.")
if free_disk < CONFIG["disk_gib"]:
    raise RuntimeError(f"This profile needs {CONFIG['disk_gib']} GiB free disk for installation and downloading.")
HARDWARE_CHECKED = True
print("Hardware check passed.")

## 3 — Install Ollama and the two Python packages

Ollama is the model server. `requests` sends HTTP requests; `ngrok` creates the
tunnel. Install the archive tools before the official Linux installer.
Colab normally has no running `systemd`; its service warning is expected because
we start the server ourselves in Step 4. A nonzero installer exit is a real error.

Run installation once per fresh runtime, before starting a server or tunnel.

In [ ]:
import sys

if not HARDWARE_CHECKED:
    raise RuntimeError("Complete the hardware check first.")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "curl", "ca-certificates", "zstd", "xz-utils"], check=True)
subprocess.run(
    ["bash", "-lc", "set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "ngrok"], check=True)
subprocess.run(["ollama", "--version"], check=True)
# A warning about no running Ollama server is expected until Step 4.

## 4 — Start Ollama and wait until it is ready

Ollama binds only to `127.0.0.1:11434` inside Colab. ngrok will connect to that
local address later. We allow one loaded model and one request at a time to limit
memory use. Model files live under `/content/ollama-models` in this temporary runtime.

The cell starts a background process, saves its log to `/content/ollama.log`, and
polls `/api/version` with timeouts. Rerunning this cell reuses a healthy server.
If startup fails, inspect that log in Colab's Files panel.

In [ ]:
import os
import time
import requests

OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": "/content/ollama-models",
    "OLLAMA_CONTEXT_LENGTH": str(NUM_CTX),
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
})

def ollama_is_ready():
    try:
        response = requests.get(f"{OLLAMA_URL}/api/version", timeout=(2, 2))
        return response.status_code == 200 and bool(response.json().get("version"))
    except (requests.RequestException, ValueError, AttributeError):
        return False

if not HARDWARE_CHECKED:
    raise RuntimeError("Complete the hardware check first.")
if not ollama_is_ready():
    old_process = globals().get("ollama_process")
    if old_process is not None and old_process.poll() is None:
        raise RuntimeError("An earlier server is still starting. Wait, or run cleanup before retrying.")
    with open("/content/ollama.log", "a", buffering=1) as log:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=OLLAMA_ENV,
        )
    for _ in range(60):
        if ollama_is_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError("Ollama exited. Inspect /content/ollama.log.")
        time.sleep(1)
    else:
        ollama_process.terminate()
        ollama_process.wait(timeout=10)
        raise RuntimeError("Ollama startup timed out. Inspect /content/ollama.log.")

print("Ollama is ready on localhost.")

## 5 — Download the selected model

`ollama pull` downloads the weights; it does not send a chat request. The first
download can take several minutes (much longer for 120B). Stay connected.
Rerunning it reuses completed files and can resume an interrupted download.
A deleted/replaced runtime loses these files.

Only the selected model is downloaded. To try another profile, follow Step 1's
cleanup instructions; the local client will list all models downloaded in this runtime.

In [ ]:
if not HARDWARE_CHECKED or not ollama_is_ready():
    raise RuntimeError("Complete the hardware and server steps first.")
subprocess.run(["ollama", "pull", MODEL], env=OLLAMA_ENV, check=True)
subprocess.run(["ollama", "list"], env=OLLAMA_ENV, check=True)

## 6 — Test a chat request inside Colab

First verify the model locally, before adding ngrok. `POST /api/chat` accepts a
model name and a list of messages. `stream: false` asks for one complete JSON
response. `num_ctx` limits context memory and `num_predict` caps output tokens.

Qwen uses `think: false` for this short exercise. GPT-OSS uses `think: "low"`,
which reduces reasoning effort; it does not disable thinking. We print only
`message.content`, the final answer. The first call includes model loading time.

In [ ]:
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Explain what an API is in two short sentences."}],
    "stream": False,
    "think": "low" if PROFILE == "instructor" else False,
    "options": {"num_ctx": NUM_CTX, "num_predict": 1024},
    "keep_alive": "15m",
}
start = time.perf_counter()
try:
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=(10, 600))
    response.raise_for_status()
    result = response.json()
except requests.Timeout:
    raise RuntimeError("Model loading/chat timed out. Check the log or use a smaller profile.") from None
except (requests.RequestException, ValueError):
    raise RuntimeError("The local chat failed. Check /content/ollama.log and the downloaded model.") from None
content = result.get("message", {}).get("content", "")
print("Model response:\n", content or "(No final text: increase num_predict or try a smaller model.)")
print(f"Execution time: {time.perf_counter() - start:.2f} seconds")
if result.get("done_reason") == "length":
    print("The token limit was reached; the answer may be incomplete.")

## 7 — Confirm that the GPU is doing the work

Look for `100% GPU` in the `PROCESSOR` column of `ollama ps`. A CPU/GPU split
can be slow; use a smaller model or context. An empty list later can simply mean
the model was unloaded after its keep-alive period; send a new chat to reload it.

In [ ]:
subprocess.run(["ollama", "ps"], env=OLLAMA_ENV, check=True)
subprocess.run(["nvidia-smi"], check=True)

## 8 — Create and verify the protected HTTPS tunnel

1. Sign in to [ngrok](https://dashboard.ngrok.com/signup) and obtain your
   [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
2. Enter it in the hidden prompt below. This token connects Colab to your ngrok
   account; **do not enter it in the local client**.
3. Create a separate, unique **API password** of 12–128 printable ASCII characters
   (letters, numbers, punctuation). Do not use `${` because ngrok treats it as a
   policy expression. Remember this password for the local client. Username: `colab`.

Never paste either secret into code, a URL, or a shell command. Keep the ngrok
authtoken in Colab only. Store the separate API password in your private local
`.env` as explained in Step 9; that file is ignored by Git. The notebook uses
secrets in memory and does not print or save them. Do not print the traffic policy or request headers.
Before sharing a notebook, clear outputs and omit cell outputs when saving.

The tunnel enforces Basic Authentication for the entire Ollama API, including
model management. Only give access to trusted people. HTTPS protects credentials
in transit; the browser-warning header used below does **not** bypass authentication.

Colab already runs an event loop. `ngrok.forward()` and `ngrok.disconnect()` can
return awaitables, so we await them before using their results; do not call
`asyncio.run()` here. Verification runs in a worker thread so it does not block
the notebook's event loop.

The cell closes earlier notebook tunnels on rerun, then checks:
**no password → 401; wrong password → 401; correct password → 200 with model JSON**.
It only prints the URL after all checks pass. On failure it attempts to close
all tunnels; if cleanup fails, disconnect and delete the runtime immediately.
If your ngrok account cannot use this policy or has reached a limit, resolve
that in the dashboard; never remove authentication to work around the error.

In [ ]:
import asyncio
import getpass
import inspect
import json
import secrets
import warnings
import ngrok

API_USER = "colab"
PUBLIC_URL = None

async def await_if_needed(value):
    return await value if inspect.isawaitable(value) else value

async def close_tunnels():
    try:
        await await_if_needed(ngrok.disconnect())
    except Exception:
        raise RuntimeError("Could not confirm tunnel closure. Disconnect and delete this Colab runtime.") from None

def verify_tunnel(url, password):
    # This Session cannot silently load credentials from .netrc.
    with requests.Session() as client:
        client.trust_env = False
        client.headers.update({"ngrok-skip-browser-warning": "true"})
        for label, auth, expected in [
            ("No credentials", None, 401),
            ("Incorrect password", (API_USER, secrets.token_urlsafe(32)), 401),
            ("Correct password", (API_USER, password), 200),
        ]:
            response = client.get(
                f"{url}/api/tags", auth=auth, timeout=(10, 30), allow_redirects=False,
            )
            print(f"{label}: HTTP {response.status_code} (expected {expected})")
            if response.status_code != expected:
                raise RuntimeError("Authentication verification failed.")
        data = response.json()
        if not isinstance(data, dict) or not isinstance(data.get("models"), list):
            raise RuntimeError("The endpoint did not return an Ollama model list.")
        if MODEL not in [item.get("name") for item in data["models"] if isinstance(item, dict)]:
            raise RuntimeError("The selected model is missing from the remote server.")

async def create_protected_tunnel():
    await close_tunnels()
    token = password = policy = None
    success = False
    stage = "secret entry"
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", getpass.GetPassWarning)
            token = getpass.getpass("ngrok authtoken (hidden): ").strip()
            password = getpass.getpass("Create an API password (hidden, 12–128 ASCII characters): ")
        if not token or not 12 <= len(password) <= 128:
            raise ValueError("Missing token or invalid password length.")
        if not password.isascii() or not password.isprintable() or "${" in password:
            raise ValueError("Invalid password characters.")
        policy = {"on_http_request": [{"actions": [
            {"type": "basic-auth", "config": {
                "realm": "CS415 Ollama", "credentials": [f"{API_USER}:{password}"], "enforce": True,
            }},
            {"type": "add-headers", "config": {"headers": {"host": "localhost"}}},
        ]}]}
        stage = "ngrok connection (check the authtoken, account policy access, and quota)"
        listener = await await_if_needed(ngrok.forward(
            "127.0.0.1:11434", authtoken=token, schemes=["HTTPS"],
            traffic_policy=json.dumps(policy),
        ))
        url = listener.url()
        if not url.startswith("https://"):
            raise RuntimeError("HTTPS is required.")
        stage = "authentication verification (check Ollama, the model, and ngrok)"
        await asyncio.to_thread(verify_tunnel, url, password)
        success = True
        return listener, url
    except Exception:
        # SDK errors can contain configuration: never display the original exception.
        raise RuntimeError(f"Setup failed during {stage}. Check Step 8's instructions and retry.") from None
    finally:
        token = password = policy = None
        if not success:
            await close_tunnels()

listener, PUBLIC_URL = await create_protected_tunnel()
print("Protected API URL:", PUBLIC_URL)
print("Username:", API_USER)
print("Use the API password you just created when running the local script.")

## Cleanup — run after the local test

**Run this cell only after finishing Step 9 on your computer.** Do not use
“Run all” for the entire notebook: that would close your tunnel before local testing.
This cell awaits ngrok shutdown and stops the Ollama process started here.
Then choose **Runtime → Disconnect and delete runtime** to release the GPU and
remove temporary files. Also use that menu if any cleanup operation fails.

In [ ]:
if "ngrok" in globals():
    await close_tunnels()
PUBLIC_URL = None
listener = None
process = globals().get("ollama_process")
if process is not None and process.poll() is None:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=10)
print("Tunnel closed and notebook-started server stopped. Disconnect and delete the runtime when finished.")

## Troubleshooting and references

| Symptom | What to do |
|---|---|
| No GPU / insufficient memory | Select a GPU runtime or a smaller profile. An A100 40 GB is insufficient for this notebook's 120B setup. |
| Installation fails / missing `zstd` | Rerun Step 3 on a fresh runtime; distinguish installation errors from the expected systemd warning. |
| Server does not start | Inspect `/content/ollama.log`; run cleanup before restarting. |
| Missing model / out of memory | Finish the pull, check `ollama list`, or choose a smaller profile. |
| ngrok setup fails | Check the authtoken, dashboard limits, and policy access. Raw SDK errors are hidden to protect secrets. |
| `_asyncio.Task` has no `.url()` | Run the supplied cell unchanged; wait for `forward()` before calling `.url()`. |
| 401 / 403 in the local script | Use the API password from Step 8, check `OLLAMA_USERNAME=colab` and `OLLAMA_PASSWORD` in your local `.env`, plus the URL and access policy. |
| Unauthenticated access succeeds | Stop the tunnel immediately. Rerun the protected setup; do not continue testing. |
| 404 / 410 / 502 / ngrok error page | The session may have expired or Ollama stopped. Reconnect Colab and rerun setup; copy the current URL. |
| 429 | Check ngrok quota and wait before retrying. |
| Timeout / empty final answer | Check the log; try a smaller model. Reasoning may exhaust `num_predict` before a final answer. |
| Missing / invalid local configuration | Create `.env` beside `test_ollama.py` and fill in all four variables; replace example settings. |
| Configured model is unavailable | Match `OLLAMA_MODEL` in `.env` to an exact name from `ollama list` in Colab. |

The local script skips ngrok's browser warning, rejects redirects, and checks TLS
certificates. It uses a direct connection (environment proxy and `.netrc` settings
are disabled). If your network requires a proxy, ask the instructor for help.

Official references: [Ollama Linux installation](https://docs.ollama.com/linux),
[chat API](https://docs.ollama.com/api/chat), [model list API](https://docs.ollama.com/api/tags),
[thinking controls](https://docs.ollama.com/capabilities/thinking),
[Qwen 3.5](https://ollama.com/library/qwen3.5),
[GPT-OSS 120B](https://ollama.com/library/gpt-oss:120b),
[ngrok Python SDK and async behavior](https://ngrok.github.io/ngrok-python/),
[Basic Auth policy](https://ngrok.com/docs/gateway/traffic-policy/actions/basic-auth),
[Host header rewriting](https://ngrok.com/docs/gateway/traffic-policy/actions/add-headers),
[Colab FAQ](https://research.google.com/colaboratory/faq.html).

## 9 — Configure and run `test_ollama.py` on YOUR computer

Finish Step 8 and keep Colab connected. **Do not run cleanup yet.** The model
runs in Colab; this Python script runs in a terminal on your own computer.
The local client loads all settings from `.env` and has no interactive prompts.

### Get the files and create `.env`

Use `test_ollama.py` and `.env.example` in `references/ollama-colab/` in your
course copy. For a downloaded copy, use **Download raw file** on both
[test_ollama.py](https://github.com/khelsawy/CS-415/blob/main/references/ollama-colab/test_ollama.py)
and [.env.example](https://github.com/khelsawy/CS-415/blob/main/references/ollama-colab/.env.example)
and save them in the same local folder. Preserve the exact filenames. The
instructor must publish the files before GitHub downloads are available.

From the repository root, create `.env` if you do not already have one:

**macOS / Linux:**

```bash
cp references/ollama-colab/.env.example references/ollama-colab/.env
```

**Windows PowerShell:**

```powershell
Copy-Item references/ollama-colab/.env.example references/ollama-colab/.env
```

This working copy includes a placeholder-only `.env`. Edit it instead of
copying over settings you have already filled in. For downloaded files, open
a terminal in their folder and use `cp .env.example .env` (macOS/Linux) or
`Copy-Item .env.example .env` (PowerShell).

Open `.env` in your editor and set:

| Variable | What to enter |
|---|---|
| `OLLAMA_URL` | **Protected API URL** from Step 8: HTTPS base URL only, without `/api/chat`. |
| `OLLAMA_USERNAME` | `colab`, matching the username printed in Step 8. |
| `OLLAMA_PASSWORD` | API password created in Step 8, not your ngrok authtoken. |
| `OLLAMA_MODEL` | Exact model from `ollama list`: `qwen3.5:4b` for `student`, `qwen3.5:0.8b` for `small`, or `gpt-oss:120b` for `instructor`. |

The template selects `gpt-oss:120b`; change it to your downloaded model.
Replace the example URL and password before running. Quote a password containing
spaces or `#` using single or double quotes, preserving the original characters.
`.env` is plain text and private: it is ignored by Git and must never be shared
or committed. `.env.example` contains placeholders only and may be shared.

### Run from the repository root (with uv)

```bash
uv run --no-project --with requests --with python-dotenv python references/ollama-colab/test_ollama.py
```

`--no-project` keeps this standalone exercise independent of the course package.
The script supports Python 3.10+ and uses `requests` plus `python-dotenv`.

### Or run with Python and pip

Open a terminal in the folder containing `test_ollama.py` and your `.env`.

**macOS / Linux:**

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install requests python-dotenv
python test_ollama.py
```

**Windows PowerShell** (activation is unnecessary):

```powershell
py -3 -m venv .venv
.\.venv\Scripts\python.exe -m pip install requests python-dotenv
.\.venv\Scripts\python.exe test_ollama.py
```

The script always loads `.env` beside itself, even when launched from a different
folder. File values override existing environment variables with the same names;
variable expansion is disabled to preserve password characters. It validates
configuration before HTTP, checks that unauthenticated access returns 401,
authenticates, verifies `OLLAMA_MODEL` with `GET /api/tags`, and sends
`POST /api/chat` with `stream: false`. It prints the answer and elapsed time,
including network travel, model loading, and generation, without printing credentials.
It waits up to 600 seconds for chat data; the first request may be slow.

When finished, return to Colab and run the **cleanup cell above**, then disconnect
and delete the runtime. If the tunnel expires, recreate it, update your local
`.env` with the current URL/password, and rerun the script.